# Fine-Tune GNN on WCSPH Fluid Dataset
This notebook generates a WCSPH fluid physics dataset, then fine-tunes a GNN pre-trained on DeepMind's WaterDrop (MPM) data.

The pre-trained checkpoint (`checkpoints/best_model.pt`) is included in the repository.

In [ ]:
!rm -rf gnn-physics-simulator
!git clone https://github.com/MarkoMile/gnn-physics-simulator.git
%cd gnn-physics-simulator

# 1. Force downgrade PyTorch to 2.8.0 (latest version supported by PyG wheels)
!pip install torch==2.8.0+cu126 --extra-index-url https://download.pytorch.org/whl/cu126

# 2. Install PyTorch Geometric extensions using pre-built wheels for torch-2.8.0+cu126
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch-geometric

# Install utility dependencies
!pip install wandb pyyaml numba

In [ ]:
# Generate WCSPH fluid dataset (~100 train, 10 valid, 10 test trajectories)
# Each trajectory: 1000 frames, 200-600 particles, dt=0.0025, h=0.015
# Aligned with WaterDrop bounds [[0.1, 0.9], [0.1, 0.9]]
!python scripts/generate_fluid_dataset.py \
    --output data/wcsph_transfer \
    --num-train 100 \
    --num-valid 10 \
    --num-test 10 \
    --min-particles 200 \
    --max-particles 600 \
    --seed 42 \
    --quiet

In [ ]:
# Quick sanity check on generated data
import numpy as np
import json

with open('data/wcsph_transfer/metadata.json') as f:
    meta = json.load(f)
    
print("Dataset metadata:")
for k, v in meta.items():
    print(f"  {k}: {v}")

# Verify a sample trajectory
data = np.load('data/wcsph_transfer/train.npz', allow_pickle=True)
traj = data[data.files[0]].item()
print(f"\nSample trajectory:")
print(f"  Position shape: {traj['position'].shape} (dtype={traj['position'].dtype})")
print(f"  Velocity shape: {traj['velocity'].shape}")
print(f"  Particle type:  unique={np.unique(traj['particle_type'])} (dtype={traj['particle_type'].dtype})")
print(f"  Position range: [{traj['position'].min():.4f}, {traj['position'].max():.4f}]")

In [ ]:
import os

# The pre-trained WaterDrop checkpoint is tracked in the git repo
checkpoint_path = 'checkpoints/best_model.pt'
assert os.path.exists(checkpoint_path), f'Checkpoint not found at {checkpoint_path}'
print(f'Checkpoint: {checkpoint_path} ({os.path.getsize(checkpoint_path) / 1e6:.1f} MB)')

In [ ]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("wandb_api_key")
    os.environ["WANDB_API_KEY"] = wandb_api_key
    
    wandb.login()
    print("WandB logged in successfully using Kaggle Secrets.")
except Exception as e:
    print(f"WandB login skipped/failed: {e}\nEnsure you have added 'wandb_api_key' to your Kaggle Secrets if you want telemetry.")

In [ ]:
# Fine-tune the GNN on the WCSPH dataset, starting from the WaterDrop checkpoint.
# Uses wcsph_transfer.yaml config
!python train.py \
    --config configs/wcsph_transfer.yaml \
    --load {checkpoint_path} \
    --wandb \
    --quiet

In [ ]:
# Copy the fine-tuned checkpoint to /kaggle/working for download
import shutil

src = 'checkpoints/best_model.pt'
dst = '/kaggle/working/best_model_wcsph.pt'

if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f"Fine-tuned model saved to {dst}")
else:
    print("No checkpoint found — training may not have completed.")